In [ ]:
print('test')

In [ ]:
# 한번 더 복습
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

response = llm.invoke('잘 응답되는지 알려줘')
print(response.content)

In [ ]:
import json
from langchain_core.documents import Document

# json 데이터 로드
with open("../storage/country_travel.json", "r", encoding="utf-8") as file:
    data = json.load(file)
print(data[:1])

In [ ]:
docs = []
for row in data:
    content = f"""
        위치: {row['name']}
        나라: {row['country']}
        설명: {row['description']}
        추천수: {row['recommendation']}
    """
    ## metadata는 검색 대상은 아니지만 문서에 딸려 다니는 부가정보를 담는곳
    ## 주로 검색 결과 필터링, 출처 원본 표시, 검색 후 가공에 쓸 값 보관
    docmake = Document(
       page_content= content.strip()
    )
    docs.append(docmake)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},  # GPU 있으면 "cuda"
    encode_kwargs={"normalize_embeddings": True} # 벡터 비교에서 중요한 건 “방향”임
)

In [ ]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="../chroma_db/travel_db"
)

In [ ]:
print(vectorstore._collection.count())

In [ ]:
vectorstore = Chroma(
    persist_directory="../chroma_db/travel_db",
    embedding_function=embeddings
)

In [ ]:
## 사용자 질문
query = "해양지 추천"
results = vectorstore.similarity_search(query, k=3)

for doc in results:
    print(doc.page_content)

In [ ]:
def search_travel(query, k=3):
    result = vectorstore.similarity_search_with_score(query, k=k)
    output = []
    for doc, distance in result:
        output.append({
            "내용": doc.page_content,
            "거리": distance
        })
    return output

query = "한국의 해양지 추천해줘"
results = search_travel(query, 5)
for r in results:
    print('거리', r['거리'])
    print('내용', r['내용'])

In [ ]:
## 거리가 짧은 데이터 검색 함수
def retrieval_docs(query, k=5):
    results = vectorstore.similarity_search(query, k = k)
    return results
## 보강
def format_docs(docs):
    data = ''
    for doc_data in docs:
        data = data + "\n\n" + doc_data.page_content
    print(data)
    return data
query = "한국 유명한 관광지 추천해줘"
format_docs(retrieval_docs(query,5))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

promt = ChatPromptTemplate.from_template("""
[역할] 여행지를 추천하고 설명과 추천수를 언급하는 여행사야
[사용자질문] {query}
[참고정보] {context}
[규칙]
- 참고정보만 참고해서 답변을 만들고 새로운 여행지를 추가하지 않는다
- 설명은 3~5줄로 한다
""")

rag_chain = (
    {
        "query": lambda x : x['usequery'],
        "context": RunnableLambda(lambda x: format_docs(retrieval_docs(x['usequery'])))
    }
    |promt
    |llm
    |StrOutputParser()
)

result = rag_chain.invoke({'usequery':'한국의 해양지 추천해줘'})
print(result)